# Retrieval-Augmented Generation (RAG)

## Goal of this Notebook

By the end of this notebook, you will:
- Understand what RAG is (intuitively, not just definition)
- Build a working RAG system step by step
- Learn embeddings, vector search, and retrieval deeply
- Gain the confidence to build your own RAG pipelines

## Chapter 1: The Problem (Why RAG Exists)
**Story**

You joined a company as an AI Engineer, and your manager asks you, "Hey, we have many internal documents. Can you build a chatbot that can answer questions on those documents? Questions like What is our refund policy?"

To build a chatbot, we can always send our questions to an LLM with a prompt, and the LLM will respond perfectly. But that didn't work for you because of the following reasons.
- LLM doesn’t know your internal docs
- Hallucinates
- Gives generic answers

**Problem:** LLM has no access to your data

<details>
  <summary><b>How will you solve this problem?</b></summary>
  <p>We will use <b>RAG (Retrieval -> Augmented -> Generation)</b>. As the name suggests, we will retrieve information from our own data, augment it with the prompt, and then provide it to the LLM to generate the response.</p>
  <img src="./img/RAG_overview.png">
</details>

## Chapter 2: Let's Read the file

So your manager provided you with a text document named "company_polices.txt" for building a basic version of a chatbot. Let's read the file

In [20]:
#reading the file company_polices.txt

with open("./company_polices.txt", "r") as f:
    content = f.read()

content[:100]

'Company Internal Policies Manual\n\n1. Code of Conduct\n\n1.1 Professional Behavior\n• Maintain respectfu'

But will it be a good idea to embed the whole file and provide it to a chatbot(LLM)?

**What can be the problems?**

<details>
    <summary><b>Problems</b></summary>
    
<details>
  <summary><b>Problem 1</b></summary>
  <h4>Context Window Limitation</h4>
  <p>Every model has a context window, which means an upper cap on how long the prompt can be. If your document is 50 pages long and the model you choose has a context window of a few thousand tokens, then the model will throw a context window exceed error</p>
</details>

<details>
  <summary><b>Problem 2</b></summary>
  <h4>Poor Retrieval Quality</h4>
  <p>If you store the entire document as one embedding, LLM will be confused. That's because:<br>
      <b>Query:</b> "What is the refund policy?"<br> but embedding represents the whole document<br><b>Result:</b><br>Model retrieves irrelevant sections (marketing, intro, etc.)and important info gets diluted
  </p>
</details>

<details>
  <summary><b>Problem 3</b></summary>
  <h4>Loss of Semantic Precision</h4>
  <p>Embeddings work best when the input is a focused area(specific information, not everything). One vector = multiple topics, and Similarity search becomes noisy</p>
</details>

<details>
  <summary><b>Problem 4</b></summary>
  <h4>Higher Cost & Latency</h4>
  <p>You send large text → expensive tokens and Slower responses</p>
</details>
</details>

**Solution**
<details>
  <summary><b>How will you solve this?</b></summary>
  <p>We will use <b>Chunking</b>. We will divide the documents into chunks and then embed each chunk separately.</p>
</details>

In [21]:
# chunking the content of the document

# here chunk_size is the size of each part we are dividing the content
# overlap is the size where 2 chunks are overlapping

def get_chunk(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

<details>
  <summary><b>Why do we use overlap?</b></summary>
  <p> When we blindly divide content into chunks, some sentences may be cut in between, which will make no sense when embedded. This will lead to loss of semantic meaning, and for some query chatbot (LLM) may not answer properly. To solve that, we intentionally duplicate some content.</p>
</details>

## Chapter 3: Let's Generate Embeddings for these chunks

Now, when we have chunks with overlay, let's generate embeddings so that we can store them for retrieval in the near future.
For this project, we are using OpenAI's ***text-embedding-ada-002*** model. Explore other models: https://developers.openai.com/api/docs/models/all

<b>You all need your API Key. Please follow the instructions given in this file:</b>
[How to get OpenAI API Key](./Get_OpenAI_API_Key.ipynb)

Now that we have api key, we will store it in a text file and import it in our environment as an environment variable

In [22]:
!pip install openai

In [23]:
import os
from openai import OpenAI

with open("openai_key.txt", "r") as file:
    openai_key = file.read()

os.environ["OPENAI_API_KEY"] = openai_key

In [24]:
def get_openai_embedding(query):
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    response = client.embeddings.create(model="text-embedding-ada-002", input=query)
    embedding = response.data[0].embedding
    return embedding

In [25]:
cat = get_openai_embedding("cat")
dog = get_openai_embedding("dog")
car = get_openai_embedding("car")

In [26]:
embd = get_openai_embedding("This is a embedding")

In [27]:
len(embd)

1536

You don't need to memorise this syntax, but eventually you will get used to it. Till then, you can refer to the official documentation here: https://developers.openai.com/api/reference/python

In [28]:
# define a function for cosine similarity

def cosine_similarity(a, b):
    similarity = 0
    for i,j in zip(a,b):
        similarity += i*j
    return similarity

In [29]:
res = cosine_similarity(cat, dog)
print(res)

0.8629749052861854


In [30]:
res = cosine_similarity(dog, car)
print(res)

0.8330534407846907


In [31]:
res = cosine_similarity(cat, car)
print(res)

0.8452008491848582


If you like to look at how and why we came to consine similarity, a full explanation you may visit these notebooks:
- [Euclidean Distance](Euclidean_Distance.ipynb)
- [Cosine Similarity](Cosine_Similarity.ipynb)

## Chapter 4: Store the chunks in Milvus

### 4.1 Install Dependencies
For this project, we will be using the Milvus vector store: https://milvus.io/ <br>
In Python, we already have packages to interact with Milvus. Let's download that


In [12]:
!pip install pymilvus
!pip install milvus_lite

Here we are downloading 2 modules:
- pymilvus (Used to interact with the Milvus vector database)
- milvus_lite (Used to run Milvus locally in a lightweight, file-based mode.)

Milvus can be deployed in two ways:
1. Server-based deployment
- Runs as a separate service (Docker/cloud)
- Suitable for production systems
- Handles large-scale data and high-performance workloads
- Can handle concurrent requests

2. File-based (Lite) deployment
- Runs locally inside your project
- Stores data in a local file (e.g., rag.db)
- No separate server required (no infrastructure required)
- Can not handel concurrent requests

For this mini-project, we use Milvus Lite (file-based setup) because: easy to set up and ideal for learning and experimentation.

### 4.2 Create connection

In [32]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("rag.db")
COLLECTION_NAME = "rag_collection"
EMBEDDING_DIM = 1536

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBEDDING_DIM,
    #metric_type="COSINE",
)

Here,
- Collection in Milvus is similar to a table in a traditional database.
- Dimension is the size (length) of each embedding vector.

Create a table named `rag_collection` that stores vectors, each with 1536 numbers. Dimension is very important, as it indicates how many dimensions the space has.

```
2D → [x, y]
3D → [x, y, z]
1536D → [x1, x2, x3, ..., x1536]
```


### 4.3 Inserting Chunks

In [33]:
def insert_chunks(chunks):
    data = []

    for i, chunk in enumerate(chunks):
        embedding = get_openai_embedding(chunk)
        data.append(
            {
                "id": i,
                "vector": embedding,
                "text": chunk,
            }
        )

    milvus_client.insert(
        collection_name=COLLECTION_NAME,
        data=data,
    )

Here we are storing the ID, vector and chunk itself. We will match the vector and then fetch the text.

## Chapter 5: Search the right chunk

In [41]:
def search(query, top_k=3):
    query_embedding = get_openai_embedding(query)

    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["text"],
    )
    return [hit["entity"]["text"] for hit in results[0]]

As told before, you don't need to memorise this syntax, but eventually you will get used to it. Till then, you can refer to the official documentation here: https://milvus.io/docs/quickstart.md

## Chapter 6: Generate the Answer

In [38]:
def generate_answer(query, context_chunks):

    system_prompt = f"""
You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not in the context, say: "I could not find that in the provided document."

Context:
{context_chunks}
"""

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
        ],
    )

    return response.choices[0].message.content

## Chapter 7: Create RAG Pipeline

Let's use our functions to create chunks, embed them and inster them inside the vector DB.

In [35]:
chunks = get_chunk(content)
insert_chunks(chunks)

In [39]:
def rag_pipeline(query):
    relevant_chunks = search(query)
    answer = generate_answer(query, relevant_chunks)
    return answer

The RAG Pipeline will take the user's query, then search relevant chunks from the vector DB and pass it to LLM with query to generate the answer.

## Chapter 8: Test it out

In [42]:
query = "What are my working hours?"
print(rag_pipeline(query, ))

Your standard working hours are 9 hours per day, with core availability required from 10 AM to 5 PM.


In [43]:
query = "Explain me leave policy."
print(rag_pipeline(query, ))

The leave policy provided states the following limits for different types of leave:

- Casual Leave: 12 days per year
- Sick Leave: 10 days per year
- Earned Leave: 15 days per year

Consequences for not following the policy:
- If an employee takes excess leave (beyond the allotted days), there will be a salary deduction.
- If an employee takes unauthorized leave, it may result in disciplinary action.

Let me know if you need more details!


I0418 21:44:07.613988  338606 chttp2_transport.cc:1369] unix:/var/folders/0y/k838qv8d5_j7mypps54lrlnw0000gn/T/tmpnppv0w22_rag.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0418 21:44:07.614245  338606 chttp2_transport.cc:1401] unix:/var/folders/0y/k838qv8d5_j7mypps54lrlnw0000gn/T/tmpnppv0w22_rag.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


<hr>
Now that your POC (Proof of Concept) is complete, the next step is to test it thoroughly using different types of queries. Once you’re confident with the results, you go back to your manager and say: `“The system works. We can now scale it to all company documents.”`

Scaling is not just “adding more documents.” You now need to make engineering decisions:
- What vector database to use
- Whether to use local or server deployment
- Which embedding model to use
- Which text generation model to use

All these decisions depend on:
- Use Case Requirements (it's a chatbot, search engine, internal tool, etc.)
- Data Size
- Latency Requirements
- Cost Constraints
- Accuracy vs Speed Tradeoff
- Infrastructure & Deployment

## Chapter 9: Final recap

**Full flow:**
1. Read the document
2. Split it into chunks with overlay
3. Embed each chunk
4. Store embedding and the respective chunk in the vector store
5. Embed the user query
6. Retrieve the most relevant chunks by matching against the query embedding
7. Pass relevant chunks to the LLM with a prompt
8. Generate an answer based on relevant chunks

<img src="./img/RAG_flow.png">

<details>
  <summary><b>When to use RAG:</b></summary>
  <p>- When LLM needs access to your private data <br>
- When the answer must stay grounded in real documents <br>
- You want better control than prompting alone <br>
- When data is large <br>
- When you need traceability <br>
- When personalisation is needed</p>
</details>
<br>
<details>
  <summary><b>When not to use RAG:</b></summary>
  <p>- When knowledge is general <br>
- When the dataset is small <br>
- When ultra-low latency is required <br>
- When exact logic is needed (not semantic) <br>
- When data is highly structured</p>
</details>

## Chapter 10: Take-Home Project

*Exercise 1:* Change the chunk size and overlap. Observe how retrieval changes. <br>
*Exercise 2:* Ask a question whose answer spans two nearby chunks. Does overlap help? <br>
*Exercise 3:* Try a question that is not in the document. Observe what the model outputs. <br>
*Exercise 4:* Try changing the system prompt and observe what happens.<br>
*Exercise 5:* Try using a different text embedding model or chat model.<br>

**Project:** Create a RAG tool that fetches relevant context. Provide this tool to the agent and refine the system prompt so the agent uses the tool whenever it needs internal knowledge. If it doesn't find anything, it responds with "no sufficient information found."